# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'expertise page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'API endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co'},
  {'type'

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
5 days ago
•
769k
•
919
Qwen/Qwen3.5-9B
Updated
3 days ago
•
172k
•
384
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
about 7 hours ago
•
674k
•
491
Qwen/Qwen3.5-27B
Updated
8 days ago
•
407k
•
569
Qwen/Qwen3.5-0.8B
Updated
2 days ago
•
93.4k
•
225
Browse 2M+ models
Spaces
Running
on
Zero
Featured
324
Omni Video Factory
🏆
324
text to video, image to video, video extend
Running
on
Zero
Featured
1.8k
Qwen Image Multiple Angles 3D Camera
🎥
1.8k
Change the camera angle of a photo with AI
Running
on
Zero
MCP
1.07k
Wan2.

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n5 days ago\n•\n769k\n•\n920\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n384\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 7 hours ago\n•\n674k\n•\n491\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n570\nQwen/Qwen3.5-0.8B\nUpdated\n2 days ago\n•\n93.4k\n•\n225\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n324\nOmni Video Factory\n🏆\n324\n

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Hugging Face: The AI community building the future

Overview
- The platform where the machine learning community collaborates on models, datasets, and applications.
- A global ecosystem designed to help you create, discover, and share AI work faster.
- Home to 2M+ models, 1M+ applications, and a thriving Spaces catalog of interactive demos.

Platform at a glance
- Models: Browse and reuse a vast library of models contributed by the community.
- Datasets: Access and contribute to 500k+ datasets to train, test, and benchmark.
- Spaces: Run and showcase AI apps and demos in a collaborative environment (examples: Omni Video Factory, Qwen Image Angles, faster-qwen3-tts, Z Image Turbo, and more).
- Applications: Explore a wide range of AI applications (Browse 1M+ applications).
- All modalities: Text, image, video, audio, and even 3D — all in one place.
- Open source stack: Move faster with Hugging Face’s open source tooling and ecosystem.
- Build your portfolio: Share your work with the world and grow your ML profile.

For customers and teams
- Enterprise-grade platform for organizations:
  - Team & Enterprise plans with scalable, secure collaboration.
  - Single Sign-On (SSO), region controls, and audit logs for governance.
  - Fine-grained access with Resource Groups and Token Management.
  - Analytics to track usage across repositories and projects.
  - Advanced compute options, including ZeroGPU, to scale ML workloads.
  - Private datasets viewer and private storage (1 TB per member, with options to expand).
  - Inference Providers and organization-wide billing with spend controls.
  - Advanced security policies and priority support from Hugging Face.
- Collaboration at scale:
  - Seamless collaboration on unlimited public and private models, datasets, and applications.
  - Private collaboration capabilities designed for teams and enterprises.
- Pricing snapshots:
  - PRO (individuals): $9 per month — enhanced storage, higher priority, Spaces Dev Mode, dataset viewer for private datasets, and a Pro badge.
  - Team: $20 per user per month — instant setup for growing teams.
  - Enterprise: Flexible options — contact sales to tailor security, governance, and compute needs.

Why Hugging Face for your organization
- Security and governance: SSO, audit logs, access controls, and policy-based security features.
- Transparency and analytics: Centralized insights on repository usage and activity.
- Scalable compute: Access to advanced compute options and ZeroGPU to accelerate workloads.
- Privacy and collaboration: Private storage and private dataset viewing for secure collaboration.
- Support and partnership: Priority support to maximize platform usage and success.

Culture and community
- The AI community building the future: Hugging Face emphasizes open collaboration, shared growth, and a community-driven approach to ML.
- A platform built for researchers, developers, data scientists, and organizations to contribute, learn, and showcase work.
- Emphasis on enabling all modalities and real-world applications, from research to production.

Careers and opportunities
- The pages you provided highlight a dynamic, forward-thinking AI community and enterprise collaboration opportunities.
- Specific job postings aren’t listed in the supplied content, but Hugging Face invites teams and individuals to join the ecosystem and work on cutting-edge AI projects.
- For those interested in joining, consider exploring opportunities within the Team/Enterprise ecosystem or pursuing your contributions to models, datasets, and Spaces that align with your interests.

Getting started
- Sign up to join the community, explore 2M+ models, 1M+ applications, and a growing Spaces catalog.
- For organizations, explore Team or Enterprise plans to empower your AI initiatives with security, governance, and scalable compute.
- If you’re focused on personal growth, consider the PRO plan to accelerate your own ML portfolio and project sharing.

Key figures (from the landing page)
- 2M+ models
- 1M+ applications
- Diverse Spaces with hundreds of active projects and demos
- A broad ecosystem spanning all ML modalities

Contact and next steps
- Visit Hugging Face to sign up or request enterprise details.
- Explore the pricing page for PRO and Team options or contact sales for Enterprise needs.
- Dive into the Models, Datasets, and Spaces to discover, contribute, and showcase your work.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face — The AI community building the future

The platform where the machine learning community collaborates on models, datasets, and applications.

## What we do
- Host and collaborate on open-source ML resources: models, datasets, and AI-powered applications.
- Open source first: move faster with the HF Open Source stack.
- Support all modalities: text, image, video, audio, and even 3D.
- Build and share your ML portfolio: publish your work and get noticed by the community.

## Platform highlights
- Models: 2M+ models available to browse and reuse.
- Datasets: 500k+ datasets for training, evaluation, and research.
- Spaces: hosted AI apps and demos to explore, deploy, and showcase.
  - Examples include text-to-video, image-to-video, 3D camera demos, speech synthesis, and more.
- Applications: 1M+ applications built on the platform, enabling rapid experimentation and deployment.

## For customers and enterprises
- Accelerate your ML initiatives with paid Compute and Enterprise solutions.
- Team & Enterprise offerings designed for collaboration, scale, and production-readiness.
- Reliable, open, and ethical AI infrastructure to support business needs.

## For developers, researchers, and the community
- Explore all modalities and build a personal ML portfolio.
- Share, discover, and collaborate on open-source ML work with a global community.
- The platform empowers the next generation of ML engineers, scientists, and end users to learn, collaborate, and shape an open and ethical AI future together.

## Culture and values
- The AI community building the future: focus on openness, collaboration, and learning.
- Empowering people to learn, contribute, and share their work openly.
- Commitment to building an open and ethical AI future.

## Brand identity
- Brand motto: Hugging Face is the collaboration platform for the machine learning community.
- Brand assets emphasize openness and community-driven innovation.
- HF color palette and branding designed to reflect energy, creativity, and accessibility.

## Get involved
- Sign up to join the community, explore models, datasets, and Spaces, and start building.
- Explore, contribute, and grow your ML career in a vibrant, open ecosystem.

## Quick start
- Explore 2M+ models, 1M+ applications, and 500k+ datasets.
- Browse Spaces to see live demos and deployable AI apps.
- Sign Up to begin collaborating and building today.

If you’d like, I can tailor this into a one-page brochure with a polished layout and add a calls-to-action section (Sign Up, Contact Us, Careers page link) based on where you plan to distribute it.

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face — The AI community building the future

Hugging Face is the collaboration platform for the machine learning community. It’s a central hub where developers, researchers, and end users share, explore, discover, and experiment with open-source ML. The mission is to learn, collaborate, and build an open and ethical AI future together.

## What we offer

- A vibrant platform to collaborate on models, datasets, and applications
  - Models, Datasets, Spaces, Community, Docs, Enterprise, Pricing
- An open-source stack that helps you move faster
- Support for all modalities: text, image, video, audio, and even 3D
- A place to build your ML portfolio and showcase your work to the world
- Easy entry to explore AI apps or browse 2M+ models

## The hub you can rely on

- 2M+ models to browse and reuse
- 1M+ applications to discover and deploy
- Spaces for running AI apps and demos (for example, Omni Video Factory and other featured spaces)
  - Examples you’ll see include Spaces focused on video, image processing, audio, and more
- Datasets hub with hundreds of thousands of datasets ready for experimentation

## Why partners choose Hugging Face

- The Home of Machine Learning: create, discover, and collaborate on ML better
- Host and collaborate on unlimited public models, datasets, and applications
- Sign up to get started quickly and accelerate your ML journey
- For teams and enterprises, we provide paid Compute and Enterprise solutions

## Our culture and values

- Open, collaborative, and community-driven: the platform empowers the next generation of ML engineers, scientists, and end users
- Open-source first: the Hub is a central place to share, explore, discover, and experiment with open-source ML
- Focused on building an open and ethical AI future together
- Welcoming to practitioners at all levels who want to learn, share, and contribute

## For customers and teams

- Flexible access to public models, datasets, and applications to accelerate projects
- Enterprise options via Team & Enterprise to scale ML workloads with paid compute
- A multi-modality platform that supports text, image, video, audio, and 3D workflows
- Build and showcase your ML portfolio to highlight your work and capabilities

## For researchers and developers

- A thriving community and ecosystem to publish and iterate on models
- Tools and spaces to experiment, validate, and demonstrate capabilities
- A straightforward path to contribute to open-source ML libraries and workflows

## Careers and opportunities

- Hugging Face emphasizes a collaborative, open-source culture and a rapidly growing community
- Specific career openings aren’t listed in the provided pages
- For opportunities, explore roles related to Team & Enterprise and join the broader ML community

## How to get started

- Explore AI Apps
- Browse 2M+ models
- Join the community and sign up to start collaborating
- If you’re looking for enterprise solutions, explore the paid Compute options and Team & Enterprise offerings

If you’re seeking a place to learn, contribute, and build in open AI, Hugging Face offers a vibrant platform and a clear path to participate in shaping the future of machine learning.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>